# El problema del viajero

El Problema del Viajante de Comercio consiste en encontrar el camino más corto para visitar un conjunto de ciudades exactamente una vez y regresar al punto de origen. En el código presentado, se modela mediante una matriz de distancias generada aleatoriamente y se resuelve con el Algoritmo de Colonia de Hormigas, donde cada hormiga construye una ruta guiada por dos factores: las feromonas depositadas en caminos previos y la distancia entre ciudades.

El objetivo principal es hallar la ruta óptima balanceando dichos factores mediante los parámetros alfa y beta. Sin embargo, la solución obtenida es una aproximación, ya que su calidad depende del número de iteraciones, la evaporación de feromonas y los parámetros elegidos. El método se inspira en el comportamiento natural de las hormigas: los caminos más cortos acumulan más feromonas, reforzando su uso futuro, mientras que la evaporación gradual evita estancarse en soluciones subóptimas. Empíricamente, el algoritmo converge hacia una solución estable entre las 200 y 300 iteraciones.

In [1]:
import random, sys, math

In [8]:
def matrizDistancias(nCiud, distanciaMaxima):
    matriz = [[0 for i in range(nCiud)] for j in range(nCiud)]

    for i in range(nCiud):
        for j in range(i):
            matriz[i][j] = distanciaMaxima*random.random()
            matriz[j][i] = matriz[i][j]

    return matriz

def eligeCiudad(dists, ferom, visitadas):
    listaPesos  = []
    disponibles = []
    actual      = visitadas[-1]

    alfa = 1.0
    beta = 0.5

    for i in range(len(dists)):
        if i not in visitadas:
            fer  = math.pow((1.0 + ferom[actual][i]), alfa)
            peso = math.pow(1.0/dists[actual][i], beta) * fer
            disponibles.append(i)
            listaPesos.append(peso)

    valor     = random.random() * sum(listaPesos)
    acumulado = 0.0
    i         = -1
    while valor > acumulado:
        i         += 1
        acumulado += listaPesos[i]

    return disponibles[i]

def eligeCamino(distancias, feromonas):
    camino     = [0]
    longCamino = 0

    while len(camino) < len(distancias):
        ciudad      = eligeCiudad(distancias, feromonas, camino)
        longCamino += distancias[camino[-1]][ciudad]
        camino.append(ciudad)

    longCamino += distancias[camino[-1]][0]
    camino.append(0)

    return (camino, longCamino)

def rastroFeromonas(feromonas, camino, dosis):
    for i in range(len(camino) - 1):
        feromonas[camino[i]][camino[i+1]] += dosis

def evaporaFeromonas(feromonas):
    for lista in feromonas:
        for i in range(len(lista)):
            lista[i] *= 0.9

def hormigas(distancias, iteraciones, distMedia):
    n         = len(distancias)
    feromonas = [[0 for i in range(n)] for j in range(n)]

    mejorCamino     = []
    longMejorCamino = sys.maxsize

    for iter in range(iteraciones):
        (camino, longCamino) = eligeCamino(distancias, feromonas)

        if longCamino <= longMejorCamino:
            mejorCamino     = camino
            longMejorCamino = longCamino

        rastroFeromonas(feromonas, camino, distMedia/longCamino)

        evaporaFeromonas(feromonas)

    return (mejorCamino, longMejorCamino)

In [9]:
numCiudades = 10
distanciaMaxima = 10
ciudades = matrizDistancias(numCiudades, distanciaMaxima)

# Obtención del mejor camino
iteraciones = 1000
distMedia = numCiudades * distanciaMaxima / 2
(camino, longCamino) = hormigas(ciudades, iteraciones, distMedia)

print("Camino: ", camino)
print("Longitud del camino: ", longCamino)

Camino:  [0, 7, 5, 1, 8, 4, 2, 3, 6, 9, 0]
Longitud del camino:  12.805170346077698
